# Predicting Newsletter Subscription from Player Characteristic 
**Introduction**

    Background

In the gaming world, keeping players engaged and interested is a key focus for game developers. One way companies can do this is through newsletters, which share updates and new events with players. Subscribing to a newsletter often shows that a player is more engaged and invested in the game community. 
By studying which players are likely to subscribe, game developers and marketers can better understand what attracts player interest. This information can help them save resources and time by focusing mainly on the characteristics that sustain player engagement. 

    Question
*Can a player’s experience level and average session time predict whether they subscribe to the game’s newsletter?*

To explore this question, we use two datasets. The players dataset includes each player’s experience level and whether or not they subscribed to the newsletter. The sessions dataset contains information about individual play sessions, including session duration. We calculate each player’s average session time using this data and then merge it with the player information.

    Dataset Description

The *players.csv* dataset contains 196 observations and 7 variables. The variables include a mix of data types:

**Object (String)**: `experience`, `hashedEmail`, `name`, `gender`

**Boolean**: `subscribe`

**Float64 (Numeric)**: `played_hours`, `age`

The dataset has 2 missing values in the age variable.

A notable characteristic of the dataset is that approximately 75% of players have played fewer than 0.6 hours, indicating a highly right-skewed distribution of gameplay time. This may affect how experience level or session-based predictions are interpreted.

Additionally, the dataset appears to be self-reported, meaning that all values rely on the players’ honesty and accuracy. As a result, the dataset may contain biases such as underreporting, overreporting, or inconsistent interpretations of survey questions, all of which can potentially limit the reliability of subsequent analysis.

The *sessions.csv* dataset contains 1535 observations and 5 variables. The variables include a mix of data types: 

**Object (String)**: `hashedEmail`,`start_time`,`end_time`

**Float64 (Numeric)**: `original_start_time`, `original_end_time`

In [ ]:
#load libraries
library(tidyverse)
library(dplyr)
library(repr)
library(tidymodels)
library(GGally)
library(ISLR)

In [ ]:
#read in players dataset
url<-"https://raw.githubusercontent.com/garyzhang25/DSI-100-Individual-Project/refs/heads/main/players.csv"
download.file(url, "players.csv")
players_data<-read_csv("players.csv")
head(players_data)

# max, min, mean of hours spent in game and the age
players_max<-players_data |>
select(played_hours, age) |>
map_df(max, na.rm=TRUE)
players_max

players_min<-players_data |>
select(played_hours, age) |>
map_df(min, na.rm=TRUE)
players_min

players_mean<-players_data |>
select(played_hours, age) |>
map_df(mean, na.rm=TRUE)
players_mean

# Each experience level and their count 
players_experience_count<-players_data |>
group_by(experience) |>
summarize(count=n()) |>
arrange(by=desc(count)) 
players_experience_count

# Proportion of subscribers
players_sub_count<- players_data |>
group_by(subscribe) |>
summarize(count=n()) |>
arrange(by=desc(count))
players_sub_count

#genders and count
players_gender_count<- players_data |>
group_by(gender) |>
summarize(count=n()) |>
arrange(by=desc(count))
players_gender_count

In [ ]:
url<-"https://raw.githubusercontent.com/garyzhang25/DSI-100-Individual-Project/refs/heads/main/sessions.csv"
download.file(url,"sessions.csv")
sessions_data<-read_csv("sessions.csv")
head(sessions_data)

session_times <- sessions_data |>
  select(start_time, end_time) |>
  separate(start_time,
           into = c("start_date", "start_time"),
           sep = " ",
           convert = TRUE) |>
  separate(end_time, 
           into = c("end_date", "end_time"),
           sep = " ",
           convert = TRUE) |>
  separate(start_time,  
           into = c("start_hour", "start_minute"),
           sep = ":", 
           convert = TRUE) |>
  separate(end_time, 
           into = c("end_hour", "end_minute"),
           sep = ":",
           convert = TRUE) |>
  mutate(start_minute_of_day = start_hour*60 + start_minute) |>
  mutate(end_minute_of_day = end_hour*60 + end_minute) |>
  mutate(session_minutes = end_minute_of_day - start_minute_of_day) |>
  select(session_minutes) |>
  filter(session_minutes >= 0) 
head(session_times)

In [ ]:
# max, min, mean of seession time
session_times_max<-session_times |>
map_df(max, na.rm=TRUE)
session_times_max

session_times_min<-session_times |>
map_df(min, na.rm=TRUE)
session_times_min

session_times_average<-session_times |>
map_df(mean, na.rm=TRUE)
session_times_average

In [ ]:
#combined data frames and added session time
merged_data <- merge(sessions_data, players_data,by = "hashedEmail")|>
    separate(start_time,
           into = c("start_date", "start_time"),
           sep = " ",
           convert = TRUE) |>
  separate(end_time, 
           into = c("end_date", "end_time"),
           sep = " ",
           convert = TRUE) |>
  separate(start_time,  
           into = c("start_hour", "start_minute"),
           sep = ":", 
           convert = TRUE) |>
  separate(end_time, 
           into = c("end_hour", "end_minute"),
           sep = ":",
           convert = TRUE) |>
  mutate(start_minute_of_day = start_hour*60 + start_minute) |>
  mutate(end_minute_of_day = end_hour*60 + end_minute) |>
  mutate(session_minutes = end_minute_of_day - start_minute_of_day) |>
  filter(session_minutes >= 0) |>
  select(name, session_minutes, experience, subscribe)|>
  group_by(name) |>
  summarize(session_time = mean(session_minutes))

#one name per session_time, select useful columns, and convert subscribe to factor type
clean_data <- merge(players_data, merged_data, by = "name")|>
  select(experience, session_time, subscribe) |>
  mutate(subscribe = as_factor(subscribe))|>
  mutate(subscribe = fct_recode(subscribe, "Yes" = "TRUE", "No" = "FALSE"))
clean_data